In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, size

# 1. Khởi tạo Spark (Chế độ Read-Only để xem)
spark = SparkSession.builder \
    .appName("Final_Product_View") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.driver.extraJavaOptions", "-Djava.net.preferIPv4Stack=true") \
    .config("spark.executor.extraJavaOptions", "-Djava.net.preferIPv4Stack=true") \
    .config("spark.sql.defaultCatalog", "my_catalog") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkSessionCatalog") \
    .config("spark.sql.catalog.spark_catalog.type", "hadoop") \
    .config("spark.sql.catalog.spark_catalog.warehouse", "s3a://warehouse/") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.type", "hadoop") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://warehouse/") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

print("=======================================================")
print("             BÁO CÁO THÀNH PHẨM DATA LAKEHOUSE          ")
print("=======================================================\n")

# --- PHẦN 1: KIỂM TRA BẢNG DIM_CUSTOMERS (SCD TYPE 2) ---
print(">>> 1. BẢNG DIMENSION: KHÁCH HÀNG (SCD TYPE 2)")
df_cust = spark.table("my_catalog.default.dim_customers")
total_cust = df_cust.count()
print(f"Tổng số bản ghi: {total_cust}")

print("\n--- Top 5 dòng thay đổi lịch sử (Minh chứng SCD hoạt động) ---")
# Lọc những người có lịch sử (tức là đã từng bị đóng dòng - is_current=false)
ids_with_history = df_cust.filter("is_current = false").select("customer_id").limit(1)
if ids_with_history.count() > 0:
    sample_id = ids_with_history.collect()[0][0]
    df_cust.filter(col("customer_id") == sample_id).orderBy("start_date").show(truncate=False)
else:
    print("(Chưa có dữ liệu lịch sử nào được tìm thấy)")

# --- PHẦN 2: KIỂM TRA BẢNG FACT_ORDERS (NESTED DATA) ---
print("\n\n>>> 2. BẢNG FACT: ĐƠN HÀNG (NESTED STRUCT/ARRAY)")
df_orders = spark.table("my_catalog.default.fact_orders_nested")
total_orders = df_orders.count()
print(f"Tổng số đơn hàng: {total_orders}")

print("\n--- Cấu trúc bảng (Schema) chuẩn Big Data ---")
df_orders.printSchema()

print("\n--- Dữ liệu mẫu: Đơn hàng chứa nhiều sản phẩm ---")
df_orders.filter(size(col("line_items")) > 1) \
    .select("order_id", "order_status", "line_items") \
    .show(3, truncate=80) # truncate=80 để cắt bớt nếu dài quá, nhưng vẫn nhìn thấy struct

# --- PHẦN 3: KIỂM TRA STORAGE (VẬT LÝ) ---
print("\n\n>>> 3. LƯU TRỮ VẬT LÝ (FILES)")
print("Dữ liệu được lưu dưới định dạng Parquet + Iceberg Metadata trên MinIO S3.")
print("Đường dẫn: s3a://warehouse/default/")

25/11/30 06:45:56 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


             BÁO CÁO THÀNH PHẨM DATA LAKEHOUSE          

>>> 1. BẢNG DIMENSION: KHÁCH HÀNG (SCD TYPE 2)


25/11/30 06:45:57 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
25/11/30 06:46:00 WARN HadoopTableOperations: Error trying to recover the latest version number for s3a://warehouse/default/dim_customers/metadata/version-hint.text
java.nio.file.AccessDeniedException: s3a://warehouse/default/dim_customers/metadata: getFileStatus on s3a://warehouse/default/dim_customers/metadata: com.amazonaws.services.s3.model.AmazonS3Exception: Forbidden (Service: Amazon S3; Status Code: 403; Error Code: 403 Forbidden; Request ID: 1KXV465HG2A9B108; S3 Extended Request ID: S7BQpZ3T/A36OIDeXUHjgBKDC/mjEj7NdO5qcNAEixvR8TYGVnR8UBlvRaW1PmuhTRkCuM1cssc=; Proxy: null), S3 Extended Request ID: S7BQpZ3T/A36OIDeXUHjgBKDC/mjEj7NdO5qcNAEixvR8TYGVnR8UBlvRaW1PmuhTRkCuM1cssc=:403 Forbidden
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:255)
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:175)
	a

Py4JJavaError: An error occurred while calling o36.table.
: org.apache.iceberg.exceptions.RuntimeIOException: Failed to refresh the table
	at org.apache.iceberg.hadoop.HadoopTableOperations.refresh(HadoopTableOperations.java:126)
	at org.apache.iceberg.hadoop.HadoopTableOperations.current(HadoopTableOperations.java:83)
	at org.apache.iceberg.BaseMetastoreCatalog.loadTable(BaseMetastoreCatalog.java:49)
	at org.apache.iceberg.shaded.com.github.benmanes.caffeine.cache.BoundedLocalCache.lambda$doComputeIfAbsent$14(BoundedLocalCache.java:2406)
	at java.base/java.util.concurrent.ConcurrentHashMap.compute(ConcurrentHashMap.java:1916)
	at org.apache.iceberg.shaded.com.github.benmanes.caffeine.cache.BoundedLocalCache.doComputeIfAbsent(BoundedLocalCache.java:2404)
	at org.apache.iceberg.shaded.com.github.benmanes.caffeine.cache.BoundedLocalCache.computeIfAbsent(BoundedLocalCache.java:2387)
	at org.apache.iceberg.shaded.com.github.benmanes.caffeine.cache.LocalCache.computeIfAbsent(LocalCache.java